# Precision Testing for Data Type Optimization

Testing whether we can safely use Float32/Int32 instead of Float64/Int64 for orderbook data.


In [1]:
import polars as pl
import numpy as np
from pathlib import Path
from datetime import date


## Load Sample Data from Each Instrument Type


In [2]:
data_root = Path('data/okx/orderbook/BTC-USD')

# Load sample files (1 day each)
swap_path = data_root / 'SWAP' / '2025-09-01.parquet'
futures_path = data_root / 'FUTURES' / '2025-09-01.parquet'
option_path = data_root / 'OPTION' / '2025-09-01.parquet'

samples = {}
if swap_path.exists():
    samples['SWAP'] = pl.scan_parquet(swap_path)
if futures_path.exists():
    samples['FUTURES'] = pl.scan_parquet(futures_path)
if option_path.exists():
    samples['OPTION'] = pl.scan_parquet(option_path)

print(f"Loaded instrument types: {list(samples.keys())}")


Loaded instrument types: ['SWAP', 'FUTURES', 'OPTION']


## Analyze Value Ranges

Check min/max for each column type across all instruments.


In [3]:
def analyze_ranges(lf: pl.LazyFrame, inst_type: str):
    """Analyze value ranges for different column types."""
    
    # Get all columns by type
    schema = lf.collect_schema()
    price_cols = [c for c in schema.names() if '_px' in c]
    qty_cols = [c for c in schema.names() if '_qty' in c]
    cnt_cols = [c for c in schema.names() if 'ordCnt' in c]
    
    results = {'inst_type': inst_type}
    
    # Check prices
    if price_cols:
        price_stats = lf.select([
            pl.min_horizontal(price_cols).alias('min_price'),
            pl.max_horizontal(price_cols).alias('max_price')
        ]).collect()
        results['min_price'] = price_stats['min_price'][0]
        results['max_price'] = price_stats['max_price'][0]
    
    # Check quantities
    if qty_cols:
        qty_stats = lf.select([
            pl.min_horizontal(qty_cols).alias('min_qty'),
            pl.max_horizontal(qty_cols).alias('max_qty')
        ]).collect()
        results['min_qty'] = qty_stats['min_qty'][0]
        results['max_qty'] = qty_stats['max_qty'][0]
    
    # Check counts
    if cnt_cols:
        cnt_stats = lf.select([
            pl.min_horizontal(cnt_cols).alias('min_cnt'),
            pl.max_horizontal(cnt_cols).alias('max_cnt')
        ]).collect()
        results['min_cnt'] = cnt_stats['min_cnt'][0]
        results['max_cnt'] = cnt_stats['max_cnt'][0]
    
    return results

# Analyze each instrument type
range_results = []
for inst_type, lf in samples.items():
    print(f"\nAnalyzing {inst_type}...")
    result = analyze_ranges(lf, inst_type)
    range_results.append(result)
    print(result)

# Create summary DataFrame
range_df = pl.DataFrame(range_results)
range_df



Analyzing SWAP...
{'inst_type': 'SWAP', 'min_price': 108199.9, 'max_price': 108207.2, 'min_qty': 1.0, 'max_qty': 11840.0, 'min_cnt': 1, 'max_cnt': 31}

Analyzing FUTURES...
{'inst_type': 'FUTURES', 'min_price': 108289.9, 'max_price': 108336.6, 'min_qty': 2.0, 'max_qty': 27.0, 'min_cnt': 1, 'max_cnt': 2}

Analyzing OPTION...
{'inst_type': 'OPTION', 'min_price': 0.005, 'max_price': 0.005, 'min_qty': 120.0, 'max_qty': 120.0, 'min_cnt': 1, 'max_cnt': 1}


inst_type,min_price,max_price,min_qty,max_qty,min_cnt,max_cnt
str,f64,f64,f64,f64,i64,i64
"""SWAP""",108199.9,108207.2,1.0,11840.0,1,31
"""FUTURES""",108289.9,108336.6,2.0,27.0,1,2
"""OPTION""",0.005,0.005,120.0,120.0,1,1


## Test Float32 Precision

Float32 has ~7 decimal digits of precision. Check if this is sufficient.


In [4]:
print("Float32 limits:")
print(f"  Max value: {np.finfo(np.float32).max:,.2f}")
print(f"  Min positive: {np.finfo(np.float32).tiny:.2e}")
print(f"  Epsilon (relative precision): {np.finfo(np.float32).eps:.2e}")
print(f"  Decimal digits of precision: ~7")

print("\nFloat64 limits:")
print(f"  Max value: {np.finfo(np.float64).max:,.2f}")
print(f"  Min positive: {np.finfo(np.float64).tiny:.2e}")
print(f"  Epsilon (relative precision): {np.finfo(np.float64).eps:.2e}")
print(f"  Decimal digits of precision: ~16")

print("\nInt32 limits:")
print(f"  Max value: {np.iinfo(np.int32).max:,}")
print(f"  Min value: {np.iinfo(np.int32).min:,}")


Float32 limits:
  Max value: 340,282,346,638,528,859,811,704,183,484,516,925,440.00
  Min positive: 1.18e-38
  Epsilon (relative precision): 1.19e-07
  Decimal digits of precision: ~7

Float64 limits:
  Max value: 179,769,313,486,231,570,814,527,423,731,704,356,798,070,567,525,844,996,598,917,476,803,157,260,780,028,538,760,589,558,632,766,878,171,540,458,953,514,382,464,234,321,326,889,464,182,768,467,546,703,537,516,986,049,910,576,551,282,076,245,490,090,389,328,944,075,868,508,455,133,942,304,583,236,903,222,948,165,808,559,332,123,348,274,797,826,204,144,723,168,738,177,180,919,299,881,250,404,026,184,124,858,368.00
  Min positive: 2.23e-308
  Epsilon (relative precision): 2.22e-16
  Decimal digits of precision: ~16

Int32 limits:
  Max value: 2,147,483,647
  Min value: -2,147,483,648


## Test Precision Loss on Actual Data

Convert a sample to Float32 and check for precision loss.


In [5]:
def test_precision_loss(lf: pl.LazyFrame, inst_type: str):
    """Test precision loss when converting to Float32."""
    
    # Take a sample
    df = lf.limit(10000).collect()
    
    # Get price columns
    price_cols = [c for c in df.columns if '_px' in c and df[c].dtype == pl.Float64]
    
    if not price_cols:
        print(f"{inst_type}: No price columns found")
        return
    
    print(f"\n{inst_type} - Testing {len(price_cols)} price columns on {len(df)} rows")
    
    for col in price_cols[:3]:  # Test first 3 columns as sample
        original = df[col].drop_nulls()
        if len(original) == 0:
            continue
            
        # Convert to float32 and back
        converted = original.cast(pl.Float32).cast(pl.Float64)
        
        # Calculate differences
        diff = (original - converted).abs()
        rel_diff = (diff / original).abs()
        
        print(f"  {col}:")
        print(f"    Max absolute diff: {diff.max():.2e}")
        print(f"    Max relative diff: {rel_diff.max():.2e}")
        print(f"    Mean relative diff: {rel_diff.mean():.2e}")
        
        # Check if any meaningful loss
        if rel_diff.max() > 1e-6:  # More than 0.0001% difference
            print(f"    ⚠️  Significant precision loss detected!")

# Test each instrument type
for inst_type, lf in samples.items():
    test_precision_loss(lf, inst_type)



SWAP - Testing 10 price columns on 10000 rows
  bid_1_px:
    Max absolute diff: 3.13e-03
    Max relative diff: 2.89e-08
    Mean relative diff: 1.35e-08
  ask_1_px:
    Max absolute diff: 3.13e-03
    Max relative diff: 2.89e-08
    Mean relative diff: 1.81e-08
  bid_2_px:
    Max absolute diff: 3.13e-03
    Max relative diff: 2.89e-08
    Mean relative diff: 6.52e-09

FUTURES - Testing 10 price columns on 10000 rows
  bid_1_px:
    Max absolute diff: 3.13e-03
    Max relative diff: 2.89e-08
    Mean relative diff: 1.75e-08
  ask_1_px:
    Max absolute diff: 3.13e-03
    Max relative diff: 2.89e-08
    Mean relative diff: 1.63e-08
  bid_2_px:
    Max absolute diff: 3.13e-03
    Max relative diff: 2.89e-08
    Mean relative diff: 1.84e-08

OPTION - Testing 10 price columns on 10000 rows
  bid_1_px:
    Max absolute diff: 1.85e-09
    Max relative diff: 5.43e-08
    Mean relative diff: nan
  ask_1_px:
    Max absolute diff: 1.42e-11
    Max relative diff: 4.75e-08
    Mean relative di

## Test Tick Size vs Float32 Precision

OKX has minimum tick sizes. Check if Float32 can represent them accurately.


In [6]:
def check_tick_precision(lf: pl.LazyFrame, inst_type: str):
    """Check if price changes are larger than Float32 precision."""
    
    # Sample data and calculate price differences
    df = lf.limit(100000).collect().sort(['symbol', 'timeMs'])
    
    if 'bid_1_px' not in df.columns:
        print(f"{inst_type}: No bid_1_px column")
        return
    
    # Calculate consecutive price differences by symbol
    price_diffs = (
        df.group_by('symbol', maintain_order=True)
        .agg([
            (pl.col('bid_1_px').diff().abs().drop_nulls()).alias('diffs')
        ])
    )
    
    # Flatten and get non-zero differences
    all_diffs = price_diffs.select(pl.col('diffs').explode()).drop_nulls()
    non_zero_diffs = all_diffs.filter(pl.col('diffs') > 0)
    
    if len(non_zero_diffs) == 0:
        print(f"{inst_type}: No price changes detected")
        return
    
    min_tick = non_zero_diffs['diffs'].min()
    median_tick = non_zero_diffs['diffs'].median()
    
    print(f"\n{inst_type} tick sizes:")
    print(f"  Min tick: {min_tick:.10f}")
    print(f"  Median tick: {median_tick:.10f}")
    
    # Check if Float32 can represent this accurately
    # Float32 epsilon at ~100k price level: ~0.01
    sample_price = df['bid_1_px'].drop_nulls().mean()
    float32_precision = sample_price * np.finfo(np.float32).eps
    
    print(f"  Avg price level: {sample_price:.2f}")
    print(f"  Float32 precision at this level: {float32_precision:.10f}")
    print(f"  Ratio (tick/precision): {min_tick / float32_precision:.2f}x")
    
    if min_tick > float32_precision * 10:  # 10x margin
        print(f"  ✅ Float32 is safe (tick >> precision)")
    else:
        print(f"  ⚠️  Float32 may not be safe (tick ~ precision)")

# Test each instrument type
for inst_type, lf in samples.items():
    check_tick_precision(lf, inst_type)



SWAP tick sizes:
  Min tick: 0.1000000000
  Median tick: 3.8500000000
  Avg price level: 107996.53
  Float32 precision at this level: 0.0128741898
  Ratio (tick/precision): 7.77x
  ⚠️  Float32 may not be safe (tick ~ precision)

FUTURES tick sizes:
  Min tick: 0.1000000000
  Median tick: 1.4000000000
  Avg price level: 108075.00
  Float32 precision at this level: 0.0128835440
  Ratio (tick/precision): 7.76x
  ⚠️  Float32 may not be safe (tick ~ precision)

OPTION tick sizes:
  Min tick: 0.0001000000
  Median tick: 0.0005000000
  Avg price level: 0.00
  Float32 precision at this level: 0.0000000004
  Ratio (tick/precision): 249605.70x
  ✅ Float32 is safe (tick >> precision)


## Calculate Storage Savings


In [7]:
def calculate_storage_savings():
    """Calculate potential storage savings."""
    
    # Per-row storage (depth=5)
    # Prices: 10 columns (5 bid + 5 ask)
    # Quantities: 10 columns
    # Counts: 10 columns
    # Time: 2 columns (timeMs, exchTimeMs)
    # Symbol: 1 column (string, ~20 bytes avg)
    
    current_bytes = (
        10 * 8 +  # prices (Float64)
        10 * 8 +  # quantities (Float64)
        10 * 8 +  # counts (Int64)
        2 * 8 +   # time (Int64)
        20        # symbol (String)
    )
    
    proposed_bytes = (
        10 * 4 +  # prices (Float32)
        10 * 4 +  # quantities (Float32)
        10 * 4 +  # counts (Int32)
        2 * 8 +   # time (Int64 - keep as-is)
        20        # symbol (String)
    )
    
    savings_pct = (1 - proposed_bytes / current_bytes) * 100
    
    print(f"Storage per row:")
    print(f"  Current (Float64/Int64): {current_bytes} bytes")
    print(f"  Proposed (Float32/Int32): {proposed_bytes} bytes")
    print(f"  Savings: {current_bytes - proposed_bytes} bytes ({savings_pct:.1f}%)")
    
    # Extrapolate to typical dataset
    rows_per_day = 100_000_000  # ~100M ticks/day for OPTIONS
    current_gb = (current_bytes * rows_per_day) / (1024**3)
    proposed_gb = (proposed_bytes * rows_per_day) / (1024**3)
    
    print(f"\nFor {rows_per_day:,} rows/day:")
    print(f"  Current: {current_gb:.2f} GB/day")
    print(f"  Proposed: {proposed_gb:.2f} GB/day")
    print(f"  Savings: {current_gb - proposed_gb:.2f} GB/day")

calculate_storage_savings()


Storage per row:
  Current (Float64/Int64): 276 bytes
  Proposed (Float32/Int32): 156 bytes
  Savings: 120 bytes (43.5%)

For 100,000,000 rows/day:
  Current: 25.70 GB/day
  Proposed: 14.53 GB/day
  Savings: 11.18 GB/day


## Decision

Based on the analysis above:

- ✅ **Float32 is SAFE** if tick sizes >> Float32 precision at typical price levels
- ✅ **Int32 is SAFE** if order counts < 2.1 billion (very unlikely)
- ⚠️ **Float32 may NOT be safe** if precision loss > 0.0001% or tick sizes ~ Float32 precision

**Recommendation:** Review the output above and decide whether to proceed with the optimization.


## Migrate Existing Data to Int32

Apply the optimization to all existing raw parquet files.


In [9]:
from okx.store import OrderbookStore

# Initialize store
store = OrderbookStore('data/okx', 'data/okx/manifest.sqlite')

# Define transformation: cast all count columns to Int32
def migrate_counts(lf: pl.LazyFrame):
    schema = lf.collect_schema()
    cnt_cols = [c for c in schema.names() if 'ordCnt' in c or 'Cnt' in c]
    if cnt_cols:
        return lf.with_columns([pl.col(c).cast(pl.Int32) for c in cnt_cols])
    return lf

# Apply to all raw files
print("Starting migration of count columns to Int32...")
store.migrate(migrate_counts, variant='raw', verbose=True)
print("\n✓ Migration complete!")


Starting migration of count columns to Int32...
[1/106] Migrating BTC-USD/SWAP/2025-09-06
[2/106] Migrating BTC-USD/SWAP/2025-09-07
[3/106] Migrating BTC-USD/SWAP/2025-09-08
[4/106] Migrating BTC-USD/SWAP/2025-09-05
[5/106] Migrating BTC-USD/SWAP/2025-09-03
[6/106] Migrating BTC-USD/SWAP/2025-09-04
[7/106] Migrating BTC-USD/SWAP/2025-09-01
[8/106] Migrating BTC-USD/SWAP/2025-09-02
[9/106] Migrating BTC-USD/SWAP/2025-09-09
[10/106] Migrating BTC-USD/SWAP/2025-09-10
[11/106] Migrating BTC-USD/SWAP/2025-09-14
[12/106] Migrating BTC-USD/SWAP/2025-09-11
[13/106] Migrating BTC-USD/SWAP/2025-09-13
[14/106] Migrating BTC-USD/SWAP/2025-09-16
[15/106] Migrating BTC-USD/SWAP/2025-09-12
[16/106] Migrating BTC-USD/SWAP/2025-09-18
[17/106] Migrating BTC-USD/SWAP/2025-09-17
[18/106] Migrating BTC-USD/SWAP/2025-09-15
[19/106] Migrating BTC-USD/SWAP/2025-09-20
[20/106] Migrating BTC-USD/SWAP/2025-09-21
[21/106] Migrating BTC-USD/SWAP/2025-09-19
[22/106] Migrating BTC-USD/SWAP/2025-09-24
[23/106] Migrat